# 02 — Engenharia de features

Duas representações saem daqui:

1. **Painel diário** (entidade × dia) e **matriz de volume** (entidade × dia de origem × horizonte),
   usada pelos modelos D+1, D+7, pico, anomalia e clusterização.
2. **Matriz por incidente**, usada pelo modelo de risco de violação de OLA.

**Regra anti-vazamento, aplicada por construção:** toda feature de uma linha usa dados até o dia de
origem `t`; o alvo é `t + h` com `h ≥ 1`. Não há janela centrada, estatística sobre a série inteira
nem índice de tempo cru.

Ao final, tudo é gravado em `dados/base.pkl`, que os notebooks 03 a 08 leem.

In [ ]:
import os
import sqlite3
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (11, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

PASTA = Path.cwd()
DADOS = PASTA / 'dados'            # saída do notebook 02, lida pelos notebooks de modelo
ARTEFATOS = PASTA / 'artefatos'    # modelos treinados por estes notebooks
DADOS.mkdir(exist_ok=True)
ARTEFATOS.mkdir(exist_ok=True)

SEMENTE = 42

## Carga e recorte do regime operacional

In [ ]:
# Banco SQLite da plataforma, já populado por `python manage.py importar_dataset`.
# Por padrão fica na raiz do projeto (pasta acima desta); a variável SQLITE_PATH sobrescreve.
BANCO = Path(os.environ.get('SQLITE_PATH', PASTA.parent / 'db.sqlite3'))
FUSO = 'America/Sao_Paulo'

SQL_INCIDENTES = """
SELECT i.id, i.numero, i.prioridade, i.categoria, i.subcategoria, i.descricao,
       i.aberto_em, i.resolvido_em, i.duracao_seg, i.aberto_por, i.status,
       i.entrou_kpi, i.kpi_violado, i.incidente_pai_id,
       p.codigo AS produto, e.nome AS equipe, c.codigo AS item_configuracao, f.slug AS familia_sinal
FROM core_incidente i
LEFT JOIN core_produto p          ON p.id = i.produto_id
LEFT JOIN core_equipe e           ON e.id = i.equipe_id
LEFT JOIN core_itemconfiguracao c ON c.id = i.item_configuracao_id
LEFT JOIN core_familiasinal f     ON f.id = i.familia_sinal_id
WHERE i.origem = 'dataset'
"""


def _para_local(serie):
    """O banco grava UTC; a operação (e o dataset) vive em horário de São Paulo."""
    return pd.to_datetime(serie, format='mixed', utc=True).dt.tz_convert(FUSO).dt.tz_localize(None)


def carregar_incidentes(banco=BANCO):
    if not Path(banco).exists():
        raise FileNotFoundError(f'Banco não encontrado em {banco}. Rode `python manage.py importar_dataset`.')
    with sqlite3.connect(f'file:{banco}?mode=ro', uri=True) as conexao:
        df = pd.read_sql_query(SQL_INCIDENTES, conexao)

    df['aberto_em'] = _para_local(df['aberto_em'])
    df['resolvido_em'] = _para_local(df['resolvido_em'])
    df['data'] = df['aberto_em'].dt.normalize()
    df['entrou_kpi'] = df['entrou_kpi'].astype(bool)
    # Nulo em quem não entrou no KPI: "não violou" é diferente de "nem foi medido".
    df['kpi_violado'] = df['kpi_violado'].astype('boolean')
    df['mttr_min'] = (df['resolvido_em'] - df['aberto_em']).dt.total_seconds() / 60
    df.loc[~df['entrou_kpi'], 'mttr_min'] = pd.NA
    for coluna in ('produto', 'equipe', 'item_configuracao', 'familia_sinal', 'categoria'):
        df[coluna] = df[coluna].fillna('').astype(str)
    return df


def detectar_inicio_regime(serie_diaria, janela=28, fracao=0.25):
    """Primeiro dia a partir do qual a mediana móvel de 28 dias nunca mais cai abaixo de 25% da
    mediana do período final. Separa operação real de carga histórica parcial (backfill)."""
    serie = pd.Series(serie_diaria).sort_index()
    referencia = serie.tail(janela * 3).median()
    if not referencia:
        return serie.index.min()
    movel = serie.rolling(janela, min_periods=janela).median()
    abaixo = movel[movel < referencia * fracao]
    if abaixo.empty:
        return serie.index.min()
    posicao = serie.index.get_loc(abaixo.index.max())
    return serie.index[min(posicao + 1, len(serie) - 1)]

In [ ]:
incidentes = carregar_incidentes()
diaria = (incidentes[incidentes['entrou_kpi']].groupby('data').size()
          .reindex(pd.date_range(incidentes['data'].min(), incidentes['data'].max()), fill_value=0))
inicio_regime = detectar_inicio_regime(diaria)
incidentes = incidentes[incidentes['data'] >= inicio_regime]
print(f'Regime operacional a partir de {inicio_regime:%d/%m/%Y}: {len(incidentes):,} incidentes')

## 1. Painel diário

Produtos, equipes, categorias, prioridades e os ativos de maior volume entram como entidades. Com
uma série global só haveria ~350 exemplos; com o painel, o mesmo mapa defasagem → alvo é aprendido
sobre dezenas de milhares de linhas, e o modelo passa a prever **por grupo**.

In [ ]:
from datetime import date, timedelta

ALVO_VOLUME = 'total_kpi'
HORIZONTE_MAX = 7
MIN_DIAS_ENTIDADE = 120
TOP_PRODUTOS, TOP_EQUIPES, TOP_CATEGORIAS, TOP_ITENS = 20, 12, 12, 8

COLUNAS_PAINEL = [
    'total', 'total_kpi', 'violacoes', 'criticos', 'resolvidos', 'soma_mttr_min',
    'p1', 'p2', 'p3', 'p4', 'p5', 'manual', 'monitoramento', 'itens_distintos', 'familias_distintas',
]
LAGS = (0, 1, 2, 3, 4, 5, 6, 7, 13, 14, 20, 21, 27, 28)
JANELAS_SOMA = (1, 2, 3, 7, 14, 30)
JANELAS_MEDIA = (3, 7, 14, 28)
DIMENSOES = ('global', 'produto', 'equipe', 'categoria', 'prioridade', 'item')


def _pascoa(ano):
    """Algoritmo de Butcher — base dos feriados móveis brasileiros."""
    a, b, c = ano % 19, ano // 100, ano % 100
    d, e = b // 4, b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19 * a + b - d - g + 15) % 30
    i, k = c // 4, c % 4
    l = (32 + 2 * e + 2 * i - h - k) % 7
    m = (a + 11 * h + 22 * l) // 451
    mes = (h + l - 7 * m + 114) // 31
    dia = ((h + l - 7 * m + 114) % 31) + 1
    return date(ano, mes, dia)


def feriados_brasil(anos):
    """Feriados nacionais fixos e móveis (o dataset não traz calendário)."""
    feriados = set()
    for ano in anos:
        pascoa = _pascoa(ano)
        feriados.update({
            date(ano, 1, 1), date(ano, 4, 21), date(ano, 5, 1), date(ano, 9, 7),
            date(ano, 10, 12), date(ano, 11, 2), date(ano, 11, 15), date(ano, 11, 20), date(ano, 12, 25),
            pascoa - timedelta(days=48), pascoa - timedelta(days=47),   # Carnaval
            pascoa - timedelta(days=2),                                 # Sexta-feira Santa
            pascoa + timedelta(days=60),                                # Corpus Christi
        })
    return feriados


def _agregar(grupo_cols, df):
    """Agregados de um recorte. MTTR e severidade contam só incidentes elegíveis a KPI."""
    elegivel = df['entrou_kpi']
    base = df.assign(
        _kpi=elegivel.astype(int),
        _violacao=df['kpi_violado'].fillna(False).astype(bool).astype(int),
        _critico=(elegivel & df['prioridade'].isin([1, 2])).astype(int),
        _resolvido=(elegivel & df['resolvido_em'].notna()).astype(int),
        _mttr=df['mttr_min'].where(elegivel).fillna(0.0),
        _manual=(df['aberto_por'] == 'manual').astype(int),
        _monitoramento=(df['aberto_por'] == 'monitoramento').astype(int),
        **{f'_p{p}': (elegivel & (df['prioridade'] == p)).astype(int) for p in range(1, 6)},
    )
    return base.groupby(grupo_cols, observed=True).agg(
        total=('numero', 'size'), total_kpi=('_kpi', 'sum'), violacoes=('_violacao', 'sum'),
        criticos=('_critico', 'sum'), resolvidos=('_resolvido', 'sum'), soma_mttr_min=('_mttr', 'sum'),
        p1=('_p1', 'sum'), p2=('_p2', 'sum'), p3=('_p3', 'sum'), p4=('_p4', 'sum'), p5=('_p5', 'sum'),
        manual=('_manual', 'sum'), monitoramento=('_monitoramento', 'sum'),
        itens_distintos=('item_configuracao', 'nunique'), familias_distintas=('familia_sinal', 'nunique'),
    ).reset_index()


def _top_entidades(df, coluna, quantidade):
    contagem = (df[df['entrou_kpi'] & (df[coluna] != '')].groupby(coluna, observed=True)['numero']
                .size().sort_values(ascending=False))
    return list(contagem.head(quantidade).index)


def painel_diario(incidentes, min_dias=MIN_DIAS_ENTIDADE):
    """Painel (dimensão, chave, data) → agregados, densificado sobre todo o calendário."""
    incidentes = incidentes.assign(prioridade_rotulo='P' + incidentes['prioridade'].astype(int).astype(str))
    partes = [_agregar(['data'], incidentes).assign(dimensao='global', chave='')]
    recortes = (
        ('produto', 'produto', TOP_PRODUTOS),
        ('equipe', 'equipe', TOP_EQUIPES),
        ('categoria', 'categoria', TOP_CATEGORIAS),
        ('prioridade', 'prioridade_rotulo', 5),
        ('item', 'item_configuracao', TOP_ITENS),
    )
    for dimensao, coluna, quantidade in recortes:
        chaves = _top_entidades(incidentes, coluna, quantidade)
        recorte = incidentes[incidentes[coluna].isin(chaves)]
        if recorte.empty:
            continue
        parte = _agregar([coluna, 'data'], recorte).rename(columns={coluna: 'chave'})
        partes.append(parte.assign(dimensao=dimensao))
    painel = pd.concat(partes, ignore_index=True)

    # Densificação: dia sem incidente é zero, não um buraco (buraco vira lag deslocado).
    calendario = pd.date_range(painel['data'].min(), painel['data'].max(), freq='D')
    indice = pd.MultiIndex.from_product(
        [pd.MultiIndex.from_frame(painel[['dimensao', 'chave']].drop_duplicates()), calendario],
        names=['entidade', 'data'],
    )
    denso = (
        painel.set_index(['dimensao', 'chave', 'data'])
        .reindex(pd.MultiIndex.from_tuples([(d, c, dt) for (d, c), dt in indice],
                                           names=['dimensao', 'chave', 'data']))
        .fillna(0).reset_index()
    )
    if min_dias:
        # Antes do primeiro dia com atividade a série é zero artificial; entidades curtas saem.
        primeiro = denso[denso['total'] > 0].groupby(['dimensao', 'chave'])['data'].min().rename('primeiro_dia')
        denso = denso.merge(primeiro, on=['dimensao', 'chave'], how='left')
        denso = denso[denso['data'] >= denso['primeiro_dia']]
        dias = denso.groupby(['dimensao', 'chave'])['data'].size().rename('dias')
        denso = denso.merge(dias, on=['dimensao', 'chave'], how='left')
        denso = denso[denso['dias'] >= min_dias].drop(columns=['primeiro_dia', 'dias'])

    denso[COLUNAS_PAINEL] = denso[COLUNAS_PAINEL].astype(float)
    denso['mttr_medio'] = np.where(denso['resolvidos'] > 0,
                                   denso['soma_mttr_min'] / denso['resolvidos'].replace(0, np.nan), 0.0)
    denso['taxa_violacao'] = np.where(denso['total_kpi'] > 0,
                                      denso['violacoes'] / denso['total_kpi'].replace(0, np.nan), 0.0)
    return denso.sort_values(['dimensao', 'chave', 'data']).reset_index(drop=True)

In [ ]:
painel = painel_diario(incidentes)
print('Entidades por dimensão:')
display(painel.groupby('dimensao')['chave'].nunique())
print(f'Linhas do painel: {len(painel):,}')
painel[painel['dimensao'] == 'global'].tail(5)[
    ['data', 'total', 'total_kpi', 'criticos', 'p2', 'p3', 'violacoes', 'mttr_medio']]

## 2. Matriz de features para previsão de volume

Famílias de features: defasagens, janelas agregadas, dinâmica/tendência, sazonalidade (mesmo dia
da semana), calendário com feriados nacionais, severidade, OLA/MTTR, alcance e escala da entidade.

In [ ]:
def _features_calendario(datas):
    """Calendário do dia PREVISTO — a única informação do futuro que já se sabe hoje."""
    dow = datas.dt.dayofweek
    feriados = feriados_brasil(range(datas.dt.year.min() - 1, datas.dt.year.max() + 2))
    return pd.DataFrame({
        'dow': dow,
        'dia_mes': datas.dt.day,
        'mes': datas.dt.month,
        'semana_ano': datas.dt.isocalendar().week.astype(int),
        'fim_de_semana': (dow >= 5).astype(int),
        'feriado': datas.dt.date.isin(feriados).astype(int),
        'vespera_feriado': (datas + pd.Timedelta(days=1)).dt.date.isin(feriados).astype(int),
        'pos_feriado': (datas - pd.Timedelta(days=1)).dt.date.isin(feriados).astype(int),
        'dow_sin': np.sin(2 * np.pi * dow / 7),
        'dow_cos': np.cos(2 * np.pi * dow / 7),
        'mes_sin': np.sin(2 * np.pi * datas.dt.month / 12),
        'mes_cos': np.cos(2 * np.pi * datas.dt.month / 12),
    }, index=datas.index)


def _historico_entidade(grupo, alvo):
    """Features de histórico de uma entidade. Tudo termina no dia de origem t (sem olhar o futuro)."""
    s = grupo[alvo]
    saida = {}
    for k in LAGS:
        saida[f'lag_{k}'] = s.shift(k)
    for janela in JANELAS_SOMA:
        saida[f'soma_{janela}'] = s.rolling(janela, min_periods=janela).sum()
    for janela in JANELAS_MEDIA:
        saida[f'media_{janela}'] = s.rolling(janela, min_periods=janela).mean()
    for janela in (7, 14, 28):
        saida[f'desvio_{janela}'] = s.rolling(janela, min_periods=janela).std()
    saida['min_7'] = s.rolling(7, min_periods=7).min()
    saida['max_7'] = s.rolling(7, min_periods=7).max()
    saida['max_28'] = s.rolling(28, min_periods=28).max()

    soma_7 = saida['soma_7']
    saida['var_7_vs_7'] = (soma_7 - soma_7.shift(7)) / soma_7.shift(7).replace(0, np.nan)
    saida['razao_7_28'] = saida['media_7'] / saida['media_28'].replace(0, np.nan)
    saida['delta_media_7_14'] = saida['media_7'] - saida['media_14']
    saida['cv_28'] = saida['desvio_28'] / saida['media_28'].replace(0, np.nan)
    saida['dias_ativos_28'] = (s > 0).rolling(28, min_periods=28).sum()
    saida['media_expandida'] = s.expanding(min_periods=7).mean()   # escala sem olhar o futuro

    operacionais = {
        'criticos': (7, 28), 'p2': (7, 28), 'p3': (7, 28), 'violacoes': (7, 28),
        'total': (7,), 'monitoramento': (7,), 'itens_distintos': (7,), 'familias_distintas': (7,),
        'mttr_medio': (7, 28), 'taxa_violacao': (7, 28), 'resolvidos': (7,),
    }
    for coluna, janelas in operacionais.items():
        for janela in janelas:
            saida[f'{coluna}_media_{janela}'] = grupo[coluna].rolling(janela, min_periods=janela).mean()

    quadro = pd.DataFrame(saida, index=grupo.index)
    quadro['share_criticos_7'] = quadro['criticos_media_7'] / quadro['media_7'].replace(0, np.nan)
    quadro['share_p2_7'] = quadro['p2_media_7'] / quadro['media_7'].replace(0, np.nan)
    quadro['share_p3_7'] = quadro['p3_media_7'] / quadro['media_7'].replace(0, np.nan)
    quadro['share_ruido_7'] = quadro['monitoramento_media_7'] / quadro['total_media_7'].replace(0, np.nan)
    quadro['delta_mttr'] = quadro['mttr_medio_media_7'] - quadro['mttr_medio_media_28']
    quadro['delta_taxa_violacao'] = quadro['taxa_violacao_media_7'] - quadro['taxa_violacao_media_28']
    return quadro


def historico_painel(painel, alvo=ALVO_VOLUME):
    """Painel + features de histórico por entidade (ainda sem alvo)."""
    painel = painel.sort_values(['dimensao', 'chave', 'data'])
    chaves = ['dimensao', 'chave']
    historico = (painel.groupby(chaves, group_keys=False, observed=True)
                 .apply(lambda g: _historico_entidade(g, alvo), include_groups=False))
    base = pd.concat([painel[chaves + ['data']], historico], axis=1)

    # Escala relativa ao global: distingue "produto grande com queda" de "produto pequeno".
    global_28 = base[base['dimensao'] == 'global'].set_index('data')['media_28'].rename('global_media_28')
    base = base.merge(global_28, left_on='data', right_index=True, how='left')
    base['share_global_28'] = base['media_28'] / base['global_media_28'].replace(0, np.nan)
    base = base.drop(columns=['global_media_28'])
    for dimensao in DIMENSOES:
        base[f'dim_{dimensao}'] = (base['dimensao'] == dimensao).astype(int)
    return base


def construir_features(painel, horizontes=range(1, HORIZONTE_MAX + 1), alvo=ALVO_VOLUME):
    """Matriz de treino: uma linha por (entidade, dia de origem, horizonte), alvo em t + h."""
    chaves = ['dimensao', 'chave']
    painel = painel.sort_values(chaves + ['data'])
    base = historico_painel(painel, alvo)
    alvos = painel.set_index(chaves + ['data'])[[alvo, 'violacoes', 'criticos', 'total']]
    serie = painel.set_index(chaves + ['data'])[alvo]

    linhas = []
    for h in horizontes:
        bloco = base.copy()
        bloco['horizonte'] = h
        bloco['data_origem'] = bloco['data']
        bloco['data_alvo'] = bloco['data'] + pd.Timedelta(days=h)

        # Sazonal conhecida na origem: mesmo dia da semana do alvo, 1 a 4 semanas antes.
        for semanas in (1, 2, 3, 4):
            deslocamento = 7 * semanas - h
            if deslocamento < 0:
                bloco[f'mesmo_dow_{semanas}'] = np.nan
                continue
            indice = pd.MultiIndex.from_arrays([
                bloco['dimensao'], bloco['chave'], bloco['data_origem'] - pd.Timedelta(days=deslocamento),
            ])
            bloco[f'mesmo_dow_{semanas}'] = serie.reindex(indice).to_numpy()
        bloco['media_mesmo_dow_4'] = bloco[[f'mesmo_dow_{s}' for s in (1, 2, 3, 4)]].mean(axis=1)

        bloco = pd.concat([bloco, _features_calendario(bloco['data_alvo'])], axis=1)
        bloco['dow_origem'] = bloco['data_origem'].dt.dayofweek

        futuros = alvos.reindex(pd.MultiIndex.from_arrays([bloco['dimensao'], bloco['chave'], bloco['data_alvo']]))
        bloco['alvo'] = futuros[alvo].to_numpy()
        bloco['alvo_violacoes'] = futuros['violacoes'].to_numpy()
        bloco['alvo_criticos'] = futuros['criticos'].to_numpy()
        bloco['alvo_total'] = futuros['total'].to_numpy()
        linhas.append(bloco)

    matriz = pd.concat(linhas, ignore_index=True).drop(columns=['data'])
    matriz = matriz[matriz['alvo'].notna()]
    # Linhas sem histórico completo (início de cada série) saem: imputar zero ensinaria um padrão falso.
    matriz = matriz.dropna(subset=['media_28', 'soma_30', 'mesmo_dow_4', 'media_expandida'])
    return matriz.reset_index(drop=True)


COLUNAS_IDENTIFICACAO = ['dimensao', 'chave', 'data_origem', 'data_alvo']
COLUNAS_ALVO = ['alvo', 'alvo_violacoes', 'alvo_criticos', 'alvo_total']


def colunas_de_features(matriz):
    excluir = set(COLUNAS_IDENTIFICACAO + COLUNAS_ALVO)
    return [c for c in matriz.columns if c not in excluir and matriz[c].dtype != object]

In [ ]:
matriz = construir_features(painel)
colunas = colunas_de_features(matriz)

grupos = {
    'defasagens (lags)': [c for c in colunas if c.startswith('lag_')],
    'janelas agregadas': [c for c in colunas if c.startswith(('soma_', 'media_', 'desvio_', 'min_', 'max_'))],
    'dinâmica / tendência': [c for c in colunas if c in ('var_7_vs_7', 'razao_7_28', 'delta_media_7_14', 'cv_28', 'dias_ativos_28')],
    'sazonal (mesmo dia da semana)': [c for c in colunas if c.startswith('mesmo_dow') or c == 'media_mesmo_dow_4'],
    'calendário': [c for c in colunas if c in ('dow', 'dia_mes', 'mes', 'semana_ano', 'fim_de_semana', 'feriado', 'vespera_feriado', 'pos_feriado', 'dow_sin', 'dow_cos', 'mes_sin', 'mes_cos', 'dow_origem', 'horizonte')],
    'severidade': [c for c in colunas if 'critico' in c or c.startswith(('p2_', 'p3_'))],
    'OLA e MTTR': [c for c in colunas if 'violacao' in c or 'mttr' in c],
    'alcance e ruído': [c for c in colunas if 'itens_distintos' in c or 'familias_distintas' in c or 'ruido' in c or 'monitoramento' in c],
    'escala da entidade': [c for c in colunas if c in ('media_expandida', 'share_global_28') or c.startswith('dim_')],
}
for nome, itens in grupos.items():
    print(f'{nome:>32}: {len(itens):>2}  {", ".join(itens[:5])}{"..." if len(itens) > 5 else ""}')
print(f'\n{"TOTAL":>32}: {len(colunas)} features | {len(matriz):,} linhas')
display(matriz.groupby('horizonte').size().rename('linhas por horizonte'))

## 3. Verificação de vazamento

Três checagens objetivas:

1. Toda linha tem `data_alvo > data_origem`.
2. Nenhuma feature é o alvo disfarçado: a correlação máxima com o alvo precisa ser compatível com
   defasagem, não com identidade (que daria 1,0).
3. As partições temporais, cortadas pela data do alvo, não se sobrepõem.

In [ ]:
print('1) data_alvo sempre no futuro da origem:', bool((matriz['data_alvo'] > matriz['data_origem']).all()))

X = matriz[colunas].astype(float).fillna(0.0).replace([np.inf, -np.inf], 0.0)
correlacoes = X.corrwith(matriz['alvo'].astype(float)).abs().sort_values(ascending=False)
print('\n2) Features mais correlacionadas com o alvo:')
display(correlacoes.head(8).round(3))

fim = matriz['data_alvo'].max()
inicio_teste = fim - pd.Timedelta(days=27)
inicio_validacao = inicio_teste - pd.Timedelta(days=28)
particoes = pd.DataFrame({
    'treino': [matriz.loc[matriz['data_alvo'] < inicio_validacao, 'data_alvo'].max()],
    'validação (início)': [inicio_validacao], 'teste (início)': [inicio_teste], 'fim': [fim],
})
print('3) Cortes temporais pela data do alvo:')
display(particoes)

### O que ficou deliberadamente de fora

- **Índice de tempo cru** (dias desde o início): árvore não extrapola, e um índice que só cresce
  deixa o modelo decorar o período. O nível recente (`media_28`, `media_expandida`) cumpre o papel.
- **Qualquer campo de desfecho** (duração, resolução, status, fechamento): não existe no momento da previsão.

## 4. Matriz por incidente (modelo de risco de OLA)

Uma linha por incidente elegível a KPI, com atributos do chamado, calendário, carga da equipe no
momento da abertura, taxas históricas expandidas e deslocadas, e o contexto da equipe/produto na véspera.

In [ ]:
# Metas de OLA por prioridade, em minutos (mesmo dicionário usado pela plataforma).
SLA_MINUTOS = {1: 240, 2: 240, 3: 720, 4: 1440, 5: 5760}

# Contexto da entidade anexado a cada incidente, sempre do dia ANTERIOR ao da abertura.
CONTEXTO_EQUIPE = ['media_7', 'media_28', 'razao_7_28', 'p2_media_7', 'share_criticos_7',
                   'taxa_violacao_media_28', 'violacoes_media_28', 'mttr_medio_media_7', 'delta_mttr']
CONTEXTO_PRODUTO = ['media_7', 'razao_7_28', 'taxa_violacao_media_28', 'share_criticos_7']
SUAVIZACAO = 50.0


def _taxa_expandida(df, coluna, alvo='violou', suavizacao=SUAVIZACAO):
    """Taxa histórica de violação vista só com o passado de cada linha (a linha N enxerga as N−1
    anteriores, nunca a si mesma). A suavização puxa entidades com pouca amostra para a taxa global."""
    ordenado = df.sort_values('aberto_em')
    grupo = ordenado.groupby(coluna, observed=True)[alvo]
    soma = grupo.cumsum() - ordenado[alvo]
    quantidade = grupo.cumcount()
    prior = (ordenado[alvo].cumsum() - ordenado[alvo]) / np.maximum(np.arange(len(ordenado)), 1)
    return ((soma + suavizacao * prior) / (quantidade + suavizacao)).reindex(df.index)


def _carga_recente(df, coluna, janela, rotulo):
    """Quantos incidentes a equipe/ativo já tinha na janela imediatamente anterior (closed='left')."""
    ordenado = df.sort_values('aberto_em')
    contagem = (ordenado.set_index('aberto_em').groupby(coluna, observed=True)['prioridade']
                .rolling(janela, closed='left').count().reset_index(level=0, drop=True))
    contagem.index = ordenado.index
    return contagem.reindex(df.index).fillna(0.0).rename(rotulo)


def features_incidentes(incidentes, contexto):
    """Uma linha por incidente elegível a KPI, só com o que se sabia NO INSTANTE DA ABERTURA.

    Ficam de fora duração, resolvido_em, status e código de fechamento: descrevem o desfecho e
    dariam uma AUC ótima para um modelo inútil.
    """
    df = incidentes[incidentes['entrou_kpi'] & incidentes['kpi_violado'].notna()].copy()
    df = df.sort_values('aberto_em').reset_index(drop=True)
    df['violou'] = df['kpi_violado'].astype(bool).astype(int)
    feriados = feriados_brasil(range(df['data'].dt.year.min(), df['data'].dt.year.max() + 2))
    abertura = df['aberto_em']

    quadro = pd.DataFrame({
        'prioridade': df['prioridade'].astype(float),
        'sla_minutos': df['prioridade'].map(SLA_MINUTOS).astype(float),
        'hora': abertura.dt.hour.astype(float),
        'minuto_do_dia': (abertura.dt.hour * 60 + abertura.dt.minute).astype(float),
        'dow': abertura.dt.dayofweek.astype(float),
        'dia_mes': abertura.dt.day.astype(float),
        'mes': abertura.dt.month.astype(float),
        'fim_de_semana': (abertura.dt.dayofweek >= 5).astype(float),
        'feriado': abertura.dt.date.isin(feriados).astype(float),
        'fora_horario': (~abertura.dt.hour.between(8, 18)).astype(float),
        'aberto_manual': (df['aberto_por'] == 'manual').astype(float),
        'tem_produto': (df['produto'] != '').astype(float),
        'tem_item_config': (df['item_configuracao'] != '').astype(float),
        'tem_pai': df['incidente_pai_id'].notna().astype(float),
    })
    for coluna, rotulo in (('equipe', 'taxa_violacao_equipe'), ('produto', 'taxa_violacao_produto'),
                           ('categoria', 'taxa_violacao_categoria'), ('familia_sinal', 'taxa_violacao_familia'),
                           ('item_configuracao', 'taxa_violacao_item')):
        quadro[rotulo] = _taxa_expandida(df, coluna).astype(float)

    quadro['carga_equipe_1h'] = _carga_recente(df, 'equipe', '1h', 'carga_equipe_1h')
    quadro['carga_equipe_24h'] = _carga_recente(df, 'equipe', '24h', 'carga_equipe_24h')
    quadro['volume_ci_7d'] = _carga_recente(df, 'item_configuracao', '7D', 'volume_ci_7d')
    quadro['volume_produto_24h'] = _carga_recente(df, 'produto', '24h', 'volume_produto_24h')

    vespera = df['data'] - pd.Timedelta(days=1)
    for dimensao, chave_df, colunas, prefixo in (('equipe', df['equipe'], CONTEXTO_EQUIPE, 'equipe'),
                                                 ('produto', df['produto'], CONTEXTO_PRODUTO, 'produto')):
        recorte = contexto[contexto['dimensao'] == dimensao].set_index(['chave', 'data'])
        anexo = recorte.reindex(pd.MultiIndex.from_arrays([chave_df, vespera]))[colunas]
        anexo.index = df.index
        quadro = quadro.join(anexo.add_prefix(f'{prefixo}_').astype(float))

    identificacao = df[['numero', 'aberto_em', 'data', 'equipe', 'produto', 'prioridade']].copy()
    return quadro.fillna(0.0).replace([np.inf, -np.inf], 0.0), df['violou'], identificacao

In [ ]:
contexto = historico_painel(painel)
X_ola, y_ola, ident_ola = features_incidentes(incidentes, contexto)
print(f'Incidentes elegíveis: {len(X_ola):,} | features: {X_ola.shape[1]} | '
      f'violações: {int(y_ola.sum())} ({y_ola.mean() * 100:.2f}%)')

campos_de_desfecho = {'duracao_seg', 'resolvido_em', 'status', 'mttr_min', 'kpi_violado'}
print('Nenhum campo de desfecho entre as features:', campos_de_desfecho.isdisjoint(X_ola.columns))
X_ola.head()

## 5. Gravação para os notebooks de modelo

In [ ]:
base = {
    'incidentes': incidentes,
    'painel': painel,
    'matriz': matriz,
    'contexto': contexto,
    'ola': {'X': X_ola, 'y': y_ola, 'identificacao': ident_ola},
    'inicio_regime': inicio_regime,
    'fim': incidentes['data'].max(),
}
pd.to_pickle(base, DADOS / 'base.pkl')
print('Gravado:', DADOS / 'base.pkl')